In [8]:
"""
03_linear_regression.py
Amazon E-Commerce Sales - Profit Prediction
Model: Linear Regression
"""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import os, joblib, warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection     import train_test_split, cross_val_score, KFold
from sklearn.linear_model        import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing       import StandardScaler, RobustScaler
from sklearn.metrics             import (mean_absolute_error, mean_squared_error,
                                         r2_score, mean_absolute_percentage_error)
from sklearn.pipeline            import Pipeline

INPUT_PATH = "data/feature_engineered.csv"
MODEL_DIR  = "models"
PLOT_DIR   = "outputs/linear_regression"
RESULT_DIR = "outputs"

os.makedirs(MODEL_DIR,  exist_ok=True)
os.makedirs(PLOT_DIR,   exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)

RANDOM_STATE = 42
TARGET       = 'profit'

# Helper: Save figure
def save_fig(name):
    path = os.path.join(PLOT_DIR, f"{name}.png")
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Saved plot: {path}")

# Helper: Print Metrics
def print_metrics(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2   = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100

    print(f"\n  Model: {name}")
    print(f"     MAE    : Rs. {mae:,.2f}")
    print(f"     RMSE   : Rs. {rmse:,.2f}")
    print(f"     R2     : {r2:.4f}")
    print(f"     MAPE   : {mape:.2f}%")
    return {'model': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2, 'MAPE%': mape}

# 1. Load Data
print("="*60)
print(" LOADING FEATURE-ENGINEERED DATA")
print("="*60)
df = pd.read_csv(INPUT_PATH, low_memory=False)
print(f"  Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")

# 2. Prepare Features & Target
print("\n" + "="*60)
print(" PREPARING FEATURES")
print("="*60)

DROP_COLS = [
    TARGET, 'revenue', 'log_revenue', 'log_profit',
    'order_id', 'asin', 'sku', 'style',
    'ship_postal_code', 'promotion_ids', 'promotion-ids',
    'date', 'Date'
]
DROP_COLS = [c for c in DROP_COLS if c in df.columns]

numeric_df = df.select_dtypes(include='number')
feature_cols = [c for c in numeric_df.columns if c not in DROP_COLS]

print(f"  Target  : {TARGET}")
print(f"  Features: {len(feature_cols)} columns")
print(f"  List    : {feature_cols[:12]} ...")

X = df[feature_cols].copy()
y = df[TARGET].copy()

mask = X.notna().all(axis=1) & y.notna()
X, y = X[mask], y[mask]
print(f"  Usable rows: {len(X):,}")

# 3. Train / Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
print(f"  Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}")

# 4. Build & Train Models
print("\n" + "="*60)
print(" TRAINING LINEAR REGRESSION VARIANTS")
print("="*60)

models = {
    'LinearRegression': LinearRegression(),
    'Ridge(alpha=1.0)'    : Ridge(alpha=1.0),
    'Ridge(alpha=10.0)'   : Ridge(alpha=10.0),
    'Lasso(alpha=0.1)'    : Lasso(alpha=0.1, max_iter=10000),
    'ElasticNet'      : ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=10000),
}

results = []
trained_models = {}

for name, estimator in models.items():
    pipe = Pipeline([
        ('scaler', RobustScaler()),
        ('model',  estimator)
    ])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_pred = np.maximum(y_pred, 0)

    metrics = print_metrics(name, y_test, y_pred)
    results.append(metrics)
    trained_models[name] = (pipe, y_pred)

results_df = pd.DataFrame(results).sort_values('R2', ascending=False)
best_name  = results_df.iloc[0]['model']
best_pipe, best_pred = trained_models[best_name]
joblib.dump(best_pipe, os.path.join(MODEL_DIR, 'linear_regression_best.pkl'))
print(f"\n  Best model: {best_name}  (R2={results_df.iloc[0]['R2']:.4f})")
print(f"  Saved to  : {MODEL_DIR}/linear_regression_best.pkl")

# 5. Cross-Validation
print("\n" + "="*60)
print(" 5-FOLD CROSS VALIDATION (Best LR Model)")
print("="*60)
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_r2  = cross_val_score(best_pipe, X, y, cv=kf, scoring='r2')
cv_mae = -cross_val_score(best_pipe, X, y, cv=kf, scoring='neg_mean_absolute_error')
print(f"  CV R2  : {cv_r2.mean():.4f} +/- {cv_r2.std():.4f}")
print(f"  CV MAE : Rs. {cv_mae.mean():,.2f} +/- Rs. {cv_mae.std():,.2f}")

# 6. Residual Analysis
print("\n" + "="*60)
print(" RESIDUAL ANALYSIS")
print("="*60)

residuals = y_test.values - best_pred
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f"Linear Regression Residual Analysis ({best_name})", fontsize=14)

axes[0].scatter(y_test, best_pred, alpha=0.4, color='steelblue', s=8)
lims = [min(y_test.min(), best_pred.min()), max(y_test.max(), best_pred.max())]
axes[0].plot(lims, lims, 'r--', lw=1.5, label='Perfect Fit')
axes[0].set_xlabel("Actual Profit (INR)"); axes[0].set_ylabel("Predicted Profit (INR)")
axes[0].set_title("Actual vs Predicted"); axes[0].legend()

axes[1].scatter(best_pred, residuals, alpha=0.4, color='darkorange', s=8)
axes[1].axhline(0, color='red', lw=1.5, ls='--')
axes[1].set_xlabel("Predicted Profit (INR)"); axes[1].set_ylabel("Residual (INR)")
axes[1].set_title("Residuals vs Predicted")

axes[2].hist(residuals, bins=60, color='mediumseagreen', edgecolor='white', alpha=0.85)
axes[2].set_xlabel("Residual (INR)"); axes[2].set_ylabel("Count")
axes[2].set_title("Residual Distribution")
axes[2].axvline(0, color='red', lw=1.5, ls='--')

save_fig("residual_analysis")

# 7. Coefficients
print("\n" + "="*60)
print(" FEATURE COEFFICIENTS (Best LR)")
print("="*60)

coefs = best_pipe.named_steps['model'].coef_
coef_df = pd.DataFrame({'Feature': feature_cols, 'Coefficient': coefs})
coef_df['AbsCoef'] = coef_df['Coefficient'].abs()
coef_df = coef_df.sort_values('AbsCoef', ascending=False).head(20)
print(coef_df[['Feature','Coefficient']].to_string(index=False))

plt.figure(figsize=(10, 7))
colours = ['steelblue' if c > 0 else 'tomato' for c in coef_df['Coefficient']]
plt.barh(coef_df['Feature'], coef_df['Coefficient'], color=colours, edgecolor='white')
plt.axvline(0, color='black', lw=0.8)
plt.title("Top 20 Feature Coefficients (Linear Regression)", fontsize=13)
plt.xlabel("Coefficient Value")
plt.gca().invert_yaxis()
save_fig("feature_coefficients")

# 8. Comparison Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Linear Regression Variant Comparison", fontsize=14)

x_pos = range(len(results_df))
axes[0].bar(x_pos, results_df['R2'], color='teal', edgecolor='white', alpha=0.85)
axes[0].set_xticks(x_pos); axes[0].set_xticklabels(results_df['model'], rotation=30, ha='right')
axes[0].set_ylabel("R2 Score"); axes[0].set_title("R2 Score by Model")
axes[0].grid(axis='y', alpha=0.4)

axes[1].bar(x_pos, results_df['RMSE'], color='salmon', edgecolor='white', alpha=0.85)
axes[1].set_xticks(x_pos); axes[1].set_xticklabels(results_df['model'], rotation=30, ha='right')
axes[1].set_ylabel("RMSE (INR)"); axes[1].set_title("RMSE by Model")
axes[1].grid(axis='y', alpha=0.4)

save_fig("model_comparison")

# 9. Save Results
results_df.to_csv(os.path.join(RESULT_DIR, "linear_regression_results.csv"), index=False)
print(f"\n  Results saved to: {RESULT_DIR}/linear_regression_results.csv")

print("\n" + "="*60)
print(" LINEAR REGRESSION COMPLETE")
print("="*60)
print(results_df.to_string(index=False))


 LOADING FEATURE-ENGINEERED DATA
  Loaded: 5,000 rows x 62 columns

 PREPARING FEATURES
  Target  : profit
  Features: 29 columns
  List    : ['qty', 'amount', 'day_of_week', 'day_of_month', 'month', 'quarter', 'year', 'is_weekend', 'has_promotion', 'fulfilled_by_amazon', 'b2b_flag', 'status_enc'] ...
  Usable rows: 5,000
  Train: 4,000 | Test: 1,000

 TRAINING LINEAR REGRESSION VARIANTS

  Model: LinearRegression
     MAE    : Rs. 31.42
     RMSE   : Rs. 61.25
     R2     : 0.9670
     MAPE   : 10.90%

  Model: Ridge(alpha=1.0)
     MAE    : Rs. 31.13
     RMSE   : Rs. 61.07
     R2     : 0.9672
     MAPE   : 10.64%

  Model: Ridge(alpha=10.0)
     MAE    : Rs. 29.06
     RMSE   : Rs. 59.86
     R2     : 0.9685
     MAPE   : 9.00%

  Model: Lasso(alpha=0.1)
     MAE    : Rs. 30.77
     RMSE   : Rs. 61.07
     R2     : 0.9672
     MAPE   : 10.29%

  Model: ElasticNet
     MAE    : Rs. 38.87
     RMSE   : Rs. 70.76
     R2     : 0.9560
     MAPE   : 19.62%

  Best model: Ridge(alpha=10.